# 🌲 Crop Yield Prediction — Random Forest Pipeline
**Target:** `Yield (Tonne or Bales/Hectare)` | **Dropped:** `Area (Hectare)`, `Production (Tonnes/Bales)`

---

## 📦 Step 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection    import train_test_split, cross_val_score, KFold
from sklearn.preprocessing      import RobustScaler, OneHotEncoder
from sklearn.pipeline           import Pipeline
from sklearn.compose            import ColumnTransformer
from sklearn.impute             import SimpleImputer
from sklearn.ensemble           import RandomForestRegressor
from sklearn.inspection         import permutation_importance
from sklearn.metrics            import (
    mean_absolute_error, mean_squared_error,
    r2_score, mean_absolute_percentage_error
)
from sklearn                    import set_config
set_config(display='diagram')

plt.rcParams['figure.facecolor'] = '#FAFAF5'
plt.rcParams['axes.facecolor']   = '#FAFAF5'
SEED = 42
print('✅ Libraries loaded')

## 📂 Step 2 — Load Data

In [ ]:
df = pd.read_excel('merged_crop_enriched_features.xlsx')
print(f'Shape : {df.shape}')
print(f'Target: Yield (Tonne or Bales/Hectare)')
print(f'Nulls :\n{df.isnull().sum()[df.isnull().sum()>0]}')
df.head()

## 🔧 Step 3 — Feature Engineering
> Drop `Area` and `Production` | Extract year | Create 6 new features

In [ ]:
df_fe = df.copy()

# ── Drop as instructed ────────────────────────────────────────────────────────
df_fe.drop(columns=['Area (Hectare)', 'Production (Tonnes/Bales)'], inplace=True)

# ── Drop constant/ID columns ──────────────────────────────────────────────────
df_fe.drop(columns=['State_Name', 'District_Name'], inplace=True)

# ── Extract year ──────────────────────────────────────────────────────────────
df_fe['Year'] = df_fe['Crop_Year'].str[:4].astype(int)
df_fe.drop(columns=['Crop_Year'], inplace=True)

# ── NEW FEATURE 1: Temperature × Humidity → heat stress index ────────────────
df_fe['Temp_Humidity_Index'] = df_fe['Avg_Temperature_C'] * df_fe['Humidity_pct'] / 100

# ── NEW FEATURE 2: Rainfall ÷ Temperature → water efficiency ─────────────────
df_fe['Rain_Temp_Ratio'] = df_fe['Rainfall_mm'] / (df_fe['Avg_Temperature_C'] + 1)

# ── NEW FEATURE 3: Years since 2004 → trend signal ───────────────────────────
df_fe['Years_Since_2004'] = df_fe['Year'] - 2004

# ── NEW FEATURE 4: Fertilizer usage band (bins) ───────────────────────────────
df_fe['Fertilizer_Band'] = pd.cut(
    df_fe['Fertilizer_kg_per_ha'],
    bins=[0, 50, 80, 110, 200],
    labels=['Low', 'Medium', 'High', 'Very High']
).astype(str)

# ── NEW FEATURE 5: Pest as ordinal (Low=0, Medium=1, High=2) ─────────────────
df_fe['Pest_Ordinal'] = df_fe['Pest_Disease_Incidence'].map({'Low': 0, 'Medium': 1, 'High': 2})

# ── NEW FEATURE 6: Season binary flags ───────────────────────────────────────
df_fe['Is_Kharif'] = (df_fe['Season'] == 'Kharif').astype(int)
df_fe['Is_Rabi']   = (df_fe['Season'] == 'Rabi').astype(int)

print('✅ Feature engineering complete')
print(f'Shape after engineering: {df_fe.shape}')
print(f'\nFeatures created:')
new_feats = ['Temp_Humidity_Index','Rain_Temp_Ratio','Years_Since_2004',
             'Fertilizer_Band','Pest_Ordinal','Is_Kharif','Is_Rabi']
for f in new_feats:
    print(f'  + {f}')
df_fe.head()

## ✂️ Step 4 — Define Features & Train/Test Split

In [ ]:
TARGET = 'Yield (Tonne or Bales/Hectare)'

CAT_FEATURES = [
    'Season', 'Crop', 'Soil_Type',
    'Irrigation_Type', 'Pest_Disease_Incidence', 'Fertilizer_Band'
]

NUM_FEATURES = [
    'Rainfall_mm', 'Avg_Temperature_C', 'Humidity_pct',
    'Fertilizer_kg_per_ha', 'Year', 'Years_Since_2004',
    'Temp_Humidity_Index', 'Rain_Temp_Ratio',
    'Pest_Ordinal', 'Is_Kharif', 'Is_Rabi'
]

X = df_fe[CAT_FEATURES + NUM_FEATURES]
y = df_fe[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

print(f'Total samples  : {len(X)}')
print(f'Train samples  : {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)')
print(f'Test  samples  : {len(X_test)}  ({len(X_test)/len(X)*100:.0f}%)')
print(f'Numeric features   : {len(NUM_FEATURES)}')
print(f'Categorical features: {len(CAT_FEATURES)}')

## ⚙️ Step 5 — Preprocessing + Random Forest Pipeline

In [ ]:
# ── Numeric: Median impute → RobustScaler (handles outlier yields) ────────────
num_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  RobustScaler())
])

# ── Categorical: Mode impute → OneHotEncode ───────────────────────────────────
cat_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ── Column Transformer ────────────────────────────────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipe, NUM_FEATURES),
    ('cat', cat_pipe, CAT_FEATURES)
])

# ── Full Pipeline: Preprocess → Random Forest ────────────────────────────────
rf_pipeline = Pipeline(steps=[
    ('preprocessor',  preprocessor),
    ('random_forest', RandomForestRegressor(
        n_estimators = 100,
        max_depth    = None,      # trees grow fully
        min_samples_split = 2,
        min_samples_leaf  = 1,
        random_state = SEED,
        n_jobs       = -1         # use all CPU cores
    ))
])

# ── Display sklearn pipeline diagram ─────────────────────────────────────────
rf_pipeline

## 🏋️ Step 6 — Train the Model

In [ ]:
rf_pipeline.fit(X_train, y_train)
print('✅ Model trained on', len(X_train), 'samples')
print('   Trees in forest:', rf_pipeline.named_steps['random_forest'].n_estimators)

## 📊 Step 7 — 5-Fold Cross Validation

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

cv_r2   = cross_val_score(rf_pipeline, X_train, y_train, cv=kf, scoring='r2',                          n_jobs=-1)
cv_mae  = -cross_val_score(rf_pipeline, X_train, y_train, cv=kf, scoring='neg_mean_absolute_error',    n_jobs=-1)
cv_rmse = np.sqrt(-cross_val_score(rf_pipeline, X_train, y_train, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1))

print('=' * 45)
print('       5-FOLD CROSS VALIDATION RESULTS')
print('=' * 45)
print(f'  R²   : {cv_r2.mean():.4f}  ± {cv_r2.std():.4f}')
print(f'  MAE  : {cv_mae.mean():.4f}  ± {cv_mae.std():.4f} T/Ha')
print(f'  RMSE : {cv_rmse.mean():.4f}  ± {cv_rmse.std():.4f} T/Ha')
print('=' * 45)
print(f'\nFold-wise R²: {[round(s,4) for s in cv_r2]}')

# CV scores bar chart
fig, ax = plt.subplots(figsize=(8, 4))
fold_labels = [f'Fold {i+1}' for i in range(5)]
bars = ax.bar(fold_labels, cv_r2, color='#2d6a4f', edgecolor='white', width=0.5)
ax.axhline(cv_r2.mean(), color='red', linewidth=2, linestyle='--',
           label=f'Mean R² = {cv_r2.mean():.4f}')
for bar, val in zip(bars, cv_r2):
    ax.text(bar.get_x() + bar.get_width()/2, val - 0.003,
            f'{val:.4f}', ha='center', va='top', fontsize=10,
            color='white', fontweight='bold')
ax.set_ylim(0.99, 1.001)
ax.set_title('5-Fold Cross Validation R² — Random Forest', fontweight='bold', fontsize=12)
ax.set_ylabel('R² Score')
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

## 🎯 Step 8 — Test Set Evaluation

In [ ]:
y_pred = rf_pipeline.predict(X_test)

test_r2   = r2_score(y_test, y_pred)
test_mae  = mean_absolute_error(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_mape = mean_absolute_percentage_error(y_test, y_pred) * 100

print('=' * 50)
print('        RANDOM FOREST — TEST SET RESULTS')
print('=' * 50)
print(f'  R²   (higher = better) : {test_r2:.4f}')
print(f'  MAE  (lower  = better) : {test_mae:.4f}  T/Ha')
print(f'  RMSE (lower  = better) : {test_rmse:.4f} T/Ha')
print(f'  MAPE (lower  = better) : {test_mape:.2f} %')
print('=' * 50)
print(f'\n  Interpretation:')
print(f'  On average, predictions are off by only {test_mae:.2f} T/Ha')
print(f'  The model explains {test_r2*100:.2f}% of yield variance')

## 📉 Step 9 — Actual vs Predicted + Residual Analysis

In [ ]:
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#FAFAF5')

# ── 1. Actual vs Predicted ────────────────────────────────────────────────────
axes[0].scatter(y_test, y_pred, alpha=0.4, s=20, color='#2980B9', edgecolors='none')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction line')
axes[0].set_xlabel('Actual Yield (T/Ha)')
axes[0].set_ylabel('Predicted Yield (T/Ha)')
axes[0].set_title(f'Actual vs Predicted\nR² = {test_r2:.4f}', fontweight='bold', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].spines[['top','right']].set_visible(False)
axes[0].text(0.05, 0.95, 'Closer to red line = better',
             transform=axes[0].transAxes, fontsize=8, va='top',
             bbox=dict(boxstyle='round', facecolor='#eaf4fb', alpha=0.8))

# ── 2. Residuals vs Predicted ─────────────────────────────────────────────────
axes[1].scatter(y_pred, residuals, alpha=0.4, s=20, color='#E67E22', edgecolors='none')
axes[1].axhline(0, color='red', linewidth=2, linestyle='--')
axes[1].set_xlabel('Predicted Yield (T/Ha)')
axes[1].set_ylabel('Residual (Actual − Predicted)')
axes[1].set_title(f'Residuals vs Predicted\nMAE = {test_mae:.4f} T/Ha', fontweight='bold', fontsize=12)
axes[1].spines[['top','right']].set_visible(False)
axes[1].text(0.05, 0.95, 'Random scatter = good (no bias)',
             transform=axes[1].transAxes, fontsize=8, va='top',
             bbox=dict(boxstyle='round', facecolor='#fef9e7', alpha=0.8))

# ── 3. Residuals distribution ─────────────────────────────────────────────────
axes[2].hist(residuals, bins=40, color='#8E44AD', edgecolor='white', alpha=0.85)
axes[2].axvline(0,               color='red',    linewidth=2, linestyle='--', label='Zero error')
axes[2].axvline(residuals.mean(),color='orange', linewidth=2, linestyle='-',
                label=f'Mean residual: {residuals.mean():.4f}')
axes[2].set_xlabel('Residual')
axes[2].set_ylabel('Count')
axes[2].set_title(f'Residuals Distribution\nRMSE = {test_rmse:.4f} T/Ha', fontweight='bold', fontsize=12)
axes[2].legend(fontsize=9)
axes[2].spines[['top','right']].set_visible(False)
axes[2].text(0.05, 0.95, 'Bell curve centered at 0 = unbiased',
             transform=axes[2].transAxes, fontsize=8, va='top',
             bbox=dict(boxstyle='round', facecolor='#f9ebea', alpha=0.8))

plt.suptitle('Random Forest — Model Diagnostics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 🌿 Step 10 — Feature Importance (Top 20)

In [ ]:
ohe_features   = rf_pipeline.named_steps['preprocessor'] \
                             .named_transformers_['cat'] \
                             .named_steps['ohe'] \
                             .get_feature_names_out(CAT_FEATURES).tolist()
all_feat_names = NUM_FEATURES + ohe_features

importances = rf_pipeline.named_steps['random_forest'].feature_importances_
feat_imp    = pd.Series(importances, index=all_feat_names).sort_values(ascending=False)

top20 = feat_imp.head(20).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor('#FAFAF5')

# Left: Top 20 horizontal bar
bar_colors = plt.cm.YlGn([0.3 + 0.7 * i / len(top20) for i in range(len(top20))])
bars = axes[0].barh(top20.index, top20.values, color=bar_colors, edgecolor='white')
for bar, val in zip(bars, top20.values):
    axes[0].text(val + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=8.5)
axes[0].set_title('Top 20 Feature Importances\n(Random Forest — Gini Importance)',
                  fontweight='bold', fontsize=12, pad=10)
axes[0].set_xlabel('Importance Score')
axes[0].spines[['top','right']].set_visible(False)
axes[0].set_facecolor('#FAFAF5')

# Right: Cumulative importance
cum_imp = feat_imp.values.cumsum()
axes[1].plot(range(1, len(cum_imp)+1), cum_imp, color='#2d6a4f', linewidth=2.5)
axes[1].axhline(0.95, color='red',    linewidth=1.5, linestyle='--', label='95% threshold')
axes[1].axhline(0.90, color='orange', linewidth=1.5, linestyle='--', label='90% threshold')
# Mark how many features reach 95%
n95 = int(np.argmax(cum_imp >= 0.95)) + 1
n90 = int(np.argmax(cum_imp >= 0.90)) + 1
axes[1].axvline(n95, color='red',    linewidth=1, linestyle=':')
axes[1].axvline(n90, color='orange', linewidth=1, linestyle=':')
axes[1].text(n95 + 0.5, 0.5, f'{n95} features\nfor 95%', color='red', fontsize=9)
axes[1].text(n90 + 0.5, 0.3, f'{n90} features\nfor 90%', color='orange', fontsize=9)
axes[1].set_title('Cumulative Feature Importance\n(How many features explain 90/95% of model?)',
                  fontweight='bold', fontsize=12, pad=10)
axes[1].set_xlabel('Number of Features (ranked by importance)')
axes[1].set_ylabel('Cumulative Importance')
axes[1].legend(fontsize=10)
axes[1].spines[['top','right']].set_visible(False)
axes[1].set_facecolor('#FAFAF5')

plt.suptitle('Feature Importance Analysis — Random Forest', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Top 10 features:')
print(feat_imp.head(10).round(4).to_string())

## 🔍 Step 11 — Prediction Error by Crop

In [ ]:
test_results = X_test.copy()
test_results['Actual']    = y_test.values
test_results['Predicted'] = y_pred
test_results['AbsError']  = np.abs(y_test.values - y_pred)

crop_error = test_results.groupby('Crop')['AbsError'].mean().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#FAFAF5')

# Left: MAE per crop
colors = ['#2d6a4f' if v <= crop_error.median() else '#e63946' for v in crop_error.values]
bars = axes[0].barh(crop_error.index, crop_error.values, color=colors, edgecolor='white')
axes[0].axvline(crop_error.mean(), color='black', linewidth=1.5, linestyle='--',
                label=f'Mean MAE: {crop_error.mean():.3f}')
axes[0].set_title('Mean Absolute Error per Crop\n(Green = below avg error | Red = above avg error)',
                  fontweight='bold', fontsize=11, pad=8)
axes[0].set_xlabel('Mean Absolute Error (T/Ha)')
axes[0].legend(fontsize=9)
axes[0].spines[['top','right']].set_visible(False)
axes[0].set_facecolor('#FAFAF5')

# Right: Actual vs Predicted scatter colored by crop (top 6 crops)
top6 = test_results['Crop'].value_counts().head(6).index
colors6 = ['#2980B9','#E74C3C','#27AE60','#F39C12','#8E44AD','#1ABC9C']
for crop, c in zip(top6, colors6):
    sub = test_results[test_results['Crop'] == crop]
    axes[1].scatter(sub['Actual'], sub['Predicted'], alpha=0.6, s=30,
                    color=c, label=crop, edgecolors='none')
lims = [test_results[['Actual','Predicted']].min().min(),
        test_results[['Actual','Predicted']].max().max()]
axes[1].plot(lims, lims, 'k--', linewidth=1.5, label='Perfect')
axes[1].set_xlabel('Actual Yield (T/Ha)')
axes[1].set_ylabel('Predicted Yield (T/Ha)')
axes[1].set_title('Actual vs Predicted — Top 6 Crops',
                  fontweight='bold', fontsize=11, pad=8)
axes[1].legend(fontsize=8, loc='upper left')
axes[1].spines[['top','right']].set_visible(False)
axes[1].set_facecolor('#FAFAF5')

plt.suptitle('Prediction Quality by Crop', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 📋 Step 12 — Final Summary

In [ ]:
print('=' * 60)
print('        🌲 RANDOM FOREST — FINAL SUMMARY')
print('=' * 60)
print(f'  Dataset             : Tripura Crop Data')
print(f'  Total Samples       : {len(X)}')
print(f'  Train / Test        : {len(X_train)} / {len(X_test)}')
print(f'  Features Used       : {len(CAT_FEATURES + NUM_FEATURES)}')
print(f'    Numeric           : {len(NUM_FEATURES)}')
print(f'    Categorical       : {len(CAT_FEATURES)}')
print(f'  Trees in Forest     : 100')
print()
print(f'  CV  R²              : {cv_r2.mean():.4f} ± {cv_r2.std():.4f}')
print(f'  CV  RMSE            : {cv_rmse.mean():.4f} T/Ha')
print()
print(f'  Test R²             : {test_r2:.4f}')
print(f'  Test MAE            : {test_mae:.4f} T/Ha')
print(f'  Test RMSE           : {test_rmse:.4f} T/Ha')
print(f'  Test MAPE           : {test_mape:.2f} %')
print()
print(f'  Top 3 Features      :')
for i, (fname, fval) in enumerate(feat_imp.head(3).items()):
    print(f'    {i+1}. {fname} ({fval:.4f})')
print('=' * 60)